In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
#load dataset
df1 = pd.read_csv("/content/ecommerce_customer_behavior_dataset.csv")
df1.head(10)

,Order_ID,Customer_ID,Date,Age,Gender,City,Product_Category,Unit_Price,Quantity,Discount_Amount,Total_Amount,Payment_Method,Device_Type,Session_Duration_Minutes,Pages_Viewed,Is_Returning_Customer,Delivery_Time_Days,Customer_Rating
0,ORD_001337,CUST_01337,2023-01-01,27,Female,Bursa,Toys,54.28,1,0.00,54.28,Debit Card,Mobile,4,14,True,8,5
1,ORD_004885,CUST_04885,2023-01-01,42,Male,Konya,Toys,244.90,1,0.00,244.90,Credit Card,Mobile,11,3,True,3,3
2,ORD_004507,CUST_04507,2023-01-01,43,Female,Ankara,Food,48.15,5,0.00,240.75,Credit Card,Mobile,7,8,True,5,2
3,ORD_000645,CUST_00645,2023-01-01,32,Male,Istanbul,Electronics,804.06,1,229.28,574.78,Credit Card,Mobile,8,10,False,1,4
4,ORD_000690,CUST_00690,2023-01-01,40,Female,Istanbul,Sports,755.61,5,0.00,3778.05,Cash on Delivery,Desktop,21,10,True,7,4
5,ORD_000223,CUST_00223,2023-01-01,43,Female,Istanbul,Beauty,122.46,1,0.00,122.46,Credit Card,Mobile,14,9,True,9,5
6,ORD_002506,CUST_02506,2023-01-01,25,Female,Izmir,Electronics,2107.37,2,0.00,4214.74,Digital Wallet,Desktop,10,5,False,6,5
7,ORD_004540,CUST_04540,2023-01-01,44,Male,Konya,Food,213.64,2,0.00,427.28,Credit Card,Mobile,10,16,False,3,5
8,ORD_001808,CUST_01808,2023-01-02,41,Male,Istanbul,Fashion,257.62,5,62.15,1225.95,Credit Card,Tablet,24,7,False,2,4
9,ORD_003413,CUST_03413,2023-01-02,58,Male,Istanbul,Sports,1784.75,4,490.25,6648.75,Debit Card,Desktop,8,5,True,5,4


In [ ]:
#__________________Create a Data Dictionary______________

data = pd.DataFrame({
    "columns name": df1.columns,
    "data type ": df1.dtypes.astype(str),
    "non-null values":df1.count().values,
    "null values":df1.isnull().sum(),
    "unique values":df1.nunique().values,
})
data.to_csv("data_dictionary.csv", index = False)

df = pd.read_csv("data_dictionary.csv")
df.head(10)

,columns name,data type,non-null values,null values,unique values
0,Order_ID,object,5000,0,5000
1,Customer_ID,object,5000,0,5000
2,Date,object,5000,0,451
3,Age,int64,5000,0,57
4,Gender,object,5000,0,3
5,City,object,5000,0,10
6,Product_Category,object,5000,0,8
7,Unit_Price,float64,5000,0,4785
8,Quantity,int64,5000,0,5
9,Discount_Amount,float64,5000,0,1405


In [ ]:
#________________initial dataprofiling__________________

#Missing Values

missing = df1.isnull().sum().sort_values(ascending=False)
print(missing)

Order_ID                    0
Customer_ID                 0
Date                        0
Age                         0
Gender                      0
City                        0
Product_Category            0
Unit_Price                  0
Quantity                    0
Discount_Amount             0
Total_Amount                0
Payment_Method              0
Device_Type                 0
Session_Duration_Minutes    0
Pages_Viewed                0
Is_Returning_Customer       0
Delivery_Time_Days          0
Customer_Rating             0
dtype: int64


In [ ]:
# Handle Missing Values
df1['Gender'] = df1['Gender'].fillna("other")

In [ ]:
#Duplicate Records
duplicates = df1.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

Duplicate rows: 0


In [ ]:
#Inconsistent Formatting
for col in df1.select_dtypes(include='object'):
    print(col, df1[col].str.lower().str.strip().nunique())

Order_ID 5000
Customer_ID 5000
Date 451
Gender 3
City 10
Product_Category 8
Payment_Method 5
Device_Type 3


In [ ]:
#Outliers (numeric columns)

numeric_cols = df1.select_dtypes(include=np.number).columns

for col in numeric_cols:
    q1 = df1[col].quantile(0.25)
    q3 = df1[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df1[(df1[col] < q1 - 1.5*iqr) | (df1[col] > q3 + 1.5*iqr)]
    print(f"{col}: {len(outliers)} outliers")

Age: 30 outliers
Unit_Price: 513 outliers
Quantity: 0 outliers
Discount_Amount: 879 outliers
Total_Amount: 546 outliers
Session_Duration_Minutes: 124 outliers
Pages_Viewed: 14 outliers
Delivery_Time_Days: 141 outliers
Customer_Rating: 0 outliers


In [ ]:
# Standardize Date Formats

df1['Date'] = pd.to_datetime(df1['Date'], errors='coerce')
print(df1.head())

     Order_ID Customer_ID       Date  Age  Gender      City Product_Category  \
0  ORD_001337  CUST_01337 2023-01-01   27  Female     Bursa             Toys   
1  ORD_004885  CUST_04885 2023-01-01   42    Male     Konya             Toys   
2  ORD_004507  CUST_04507 2023-01-01   43  Female    Ankara             Food   
3  ORD_000645  CUST_00645 2023-01-01   32    Male  Istanbul      Electronics   
4  ORD_000690  CUST_00690 2023-01-01   40  Female  Istanbul           Sports   

   Unit_Price  Quantity  Discount_Amount  Total_Amount    Payment_Method  \
0       54.28         1             0.00         54.28        Debit Card   
1      244.90         1             0.00        244.90       Credit Card   
2       48.15         5             0.00        240.75       Credit Card   
3      804.06         1           229.28        574.78       Credit Card   
4      755.61         5             0.00       3778.05  Cash on Delivery   

  Device_Type  Session_Duration_Minutes  Pages_Viewed  Is_Retu

In [ ]:
#Fix Inconsistent Categories

df1['Gender'] = df1['Gender'].str.lower().map({
    'm': 'Male',
    'male': 'Male',
    'f': 'Female',
    'female': 'Female'
})
print(df1["Gender"])

0       Female
1         Male
2       Female
3         Male
4       Female
         ...  
4995    Female
4996      Male
4997    Female
4998    Female
4999    Female
Name: Gender, Length: 5000, dtype: object


In [ ]:
# Final Validation Checks

print("\n","null values ","\n" , df1.isnull().sum())  # ensure handled
print("\n","duplicate values ","\n" ,df1.duplicated().sum())  # should be 0
print("\n","data type ","\n" ,df1.dtypes)  # confirm correct types


 null values  
 Order_ID                     0
Customer_ID                  0
Date                         0
Age                          0
Gender                      73
City                         0
Product_Category             0
Unit_Price                   0
Quantity                     0
Discount_Amount              0
Total_Amount                 0
Payment_Method               0
Device_Type                  0
Session_Duration_Minutes     0
Pages_Viewed                 0
Is_Returning_Customer        0
Delivery_Time_Days           0
Customer_Rating              0
dtype: int64

 duplicate values  
 0

 data type  
 Order_ID                            object
Customer_ID                         object
Date                        datetime64[ns]
Age                                  int64
Gender                              object
City                                object
Product_Category                    object
Unit_Price                         float64
Quantity                     

In [ ]:
#Export Analysis-Ready Dataset

df1.to_csv("cleaned_dataset.csv", index=False)